<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/PlantVillage_Xception_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================
# 1. Kaggle Setup and Dataset Download
# ===============================
from google.colab import files
files.upload()  # Upload kaggle.json first

!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download PlantVillage dataset
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip -d PlantVillage

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
 87% 570M/658M [00:03<00:01, 62.3MB/s]
100% 658M/658M [00:03<00:00, 194MB/s] 


In [ ]:
# ===============================
# 2. Imports
# ===============================
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import Xception
from tensorflow.keras.preprocessing import image_dataset_from_directory

# Parameters
img_size = (299, 299)   # Xception default input size
batch_size = 32

In [ ]:
# ===============================
# 3. Dataset Creation
# ===============================
dataset = image_dataset_from_directory(
    "/content/PlantVillage/PlantVillage",
    image_size=img_size,
    batch_size=batch_size,
    validation_split=0.2,
    subset="training",
    seed=123
)

val_dataset = image_dataset_from_directory(
    "/content/PlantVillage/PlantVillage",
    image_size=img_size,
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    seed=123
)

# Split val -> val + test
val_batches = tf.data.experimental.cardinality(val_dataset)
test_dataset = val_dataset.take(val_batches // 2)
val_dataset = val_dataset.skip(val_batches // 2)

class_names = dataset.class_names
num_classes = len(class_names)
print("Classes:", class_names)
print("Num Classes:", num_classes)

Found 20638 files belonging to 15 classes.
Using 16511 files for training.
Found 20638 files belonging to 15 classes.
Using 4127 files for validation.
Classes: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']
Num Classes: 15


In [ ]:
# ===============================
# 4. Data Augmentation
# ===============================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

train_ds = dataset.map(lambda x, y: (data_augmentation(x), y))

# Prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_dataset = val_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [ ]:
# ===============================
# 5. Build Xception Model
# ===============================
base_model = Xception(
    include_top=False,
    weights="imagenet",
    input_shape=(299, 299, 3)
)
base_model.trainable = False  # Freeze base initially

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

# One-hot mapping
def one_hot_map(images, labels):
    return images, tf.one_hot(labels, depth=num_classes)

train_ds_onehot = train_ds.map(one_hot_map)
val_ds_onehot = val_dataset.map(one_hot_map)
test_ds_onehot = test_dataset.map(one_hot_map)

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
# ===============================
# 6. Initial Training
# ===============================
history = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=10
)

Epoch 1/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 540s 968ms/step - accuracy: 0.1344 - loss: 7.6725 - val_accuracy: 0.2126 - val_loss: 2.6828
Epoch 2/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 443s 858ms/step - accuracy: 0.1944 - loss: 2.6989 - val_accuracy: 0.2328 - val_loss: 2.4881
Epoch 3/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 440s 851ms/step - accuracy: 0.2149 - loss: 2.5693 - val_accuracy: 0.2670 - val_loss: 2.3731
Epoch 4/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 446s 860ms/step - accuracy: 0.2478 - loss: 2.4587 - val_accuracy: 0.3829 - val_loss: 2.2345
Epoch 5/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 442s 857ms/step - accuracy: 0.2742 - loss: 2.4031 - val_accuracy: 0.3540 - val_loss: 2.2209
Epoch 6/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 505s 863ms/step - accuracy: 0.2857 - loss: 2.3598 - val_accuracy: 0.3805 - val_loss: 2.1781
Epoch 7/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 450s 871ms/step - accuracy: 0.2958 - loss: 2.3338 - val_accuracy: 0.3795 - val_loss: 2.1524
Epoch 8/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 449s 870ms/step - accuracy: 0.2915 -

In [ ]:
# ===============================
# 7. Fine-Tuning
# ===============================
base_model.trainable = True
fine_tune_at = 100  # Freeze first 100 layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=20,
    initial_epoch=10
)

Epoch 11/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 561s 1s/step - accuracy: 0.1133 - loss: 2.6737 - val_accuracy: 0.2511 - val_loss: 2.4989
Epoch 12/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 484s 938ms/step - accuracy: 0.2544 - loss: 2.4794 - val_accuracy: 0.2973 - val_loss: 2.3975
Epoch 13/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 482s 934ms/step - accuracy: 0.3175 - loss: 2.3322 - val_accuracy: 0.3454 - val_loss: 2.2857
Epoch 14/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 484s 939ms/step - accuracy: 0.3571 - loss: 2.2389 - val_accuracy: 0.3911 - val_loss: 2.1973
Epoch 15/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 484s 938ms/step - accuracy: 0.3888 - loss: 2.1544 - val_accuracy: 0.4122 - val_loss: 2.0942
Epoch 16/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 492s 952ms/step - accuracy: 0.4249 - loss: 2.0900 - val_accuracy: 0.4502 - val_loss: 2.0023
Epoch 17/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 491s 931ms/step - accuracy: 0.4662 - loss: 2.0065 - val_accuracy: 0.4632 - val_loss: 1.9506
Epoch 18/20
516/516 ━━━━━━━━━━━━━━━━━━━━ 485s 940ms/step - accuracy: 0.4

In [ ]:
# ===============================
# 8. Evaluation with Temperature Scaling
# ===============================
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize

TEMPERATURE = 2.0

def predict_with_temperature(model, dataset, T=1.0):
    all_probs, all_labels = [], []
    for images, labels in dataset:
        logits = model(images, training=False)
        scaled_logits = logits / T
        probs = tf.nn.softmax(scaled_logits).numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_probs)

y_true, y_probs = predict_with_temperature(model, test_ds_onehot, T=TEMPERATURE)
y_pred = np.argmax(y_probs, axis=1)

# Convert one-hot labels to integers
y_true_int = np.argmax(y_true, axis=1)

y_true_bin = label_binarize(y_true_int, classes=range(num_classes))

accuracy = accuracy_score(y_true_int, y_pred)
precision = precision_score(y_true_int, y_pred, average="weighted")
recall = recall_score(y_true_int, y_pred, average="weighted")
f1 = f1_score(y_true_int, y_pred, average="weighted")
kappa = cohen_kappa_score(y_true_int, y_pred)
auc = roc_auc_score(y_true_bin, y_probs, average="macro", multi_class="ovr")

print("\n Evaluation Metrics After Calibration:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Kappa    : {kappa:.4f}")


 Evaluation Metrics After Calibration:
Accuracy : 0.5317
Precision: 0.4940
Recall   : 0.5317
F1 Score : 0.4665
AUC      : 0.9078
Kappa    : 0.4810


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
